# Exercicis — la forma de cada model

**Optativa d'Aprenentatge automàtic — DAM/DAW 2n**

Heu vist cinc models, tots sobre Iris: k veïns, arbre, bosc, regressió
logística i SVM. Iris té 4 columnes; no es pot dibuixar. En aquest quadern
les dades són **fabricades**, amb `sklearn.datasets`, i tenen **només dues
columnes a posta**. Amb dues columnes es pot fer una cosa que amb Iris no
es podia: **dibuixar la frontera de decisió de cada model i veure quina
forma té**.

I la forma delata el model:

- **Regressió logística**: sempre una **recta**. Si les dades no se
  separen amb una recta, fracassa i es veu.
- **Arbre**: **escales**, rectangles amb els costats paral·lels als eixos.
  Cada tall és «aquesta columna és més gran que tal número».
- **Bosc**: les mateixes escales de l'arbre, suavitzades per la votació
  de molts arbres.
- **k-NN**: fronteres irregulars que s'enganxen als punts.
- **SVM (kernel `rbf`)**: corbes suaus.

Aquest és el quadern on es veu per fi en què es diferencien els cinc
models, perquè es dibuixen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

RANDOM_STATE = 42

## Dues eines fetes

Aquestes dues funcions es fan servir a tot el quadern. **No cal escriure-les
ni entendre-les línia a línia**; és per aquí per on comencen tots els
exercicis.

`dibuixa_frontera` fa una graella fina sobre el pla, pregunta al model de
quina classe és cada punt de la graella (`predict`) i pinta el resultat amb
`contourf`. Per sobre hi dibuixa les dades de veritat.

`crea_models` retorna els cinc models de sempre, sense entrenar, amb els
mateixos hiperparàmetres a tot arreu perquè les comparacions siguin justes.

In [ ]:
def dibuixa_frontera(ax, model, X, y, titol=""):
    """Pinta la regió que `model` assigna a cada classe, amb les dades a sobre."""
    marge = 0.5
    x1_min, x1_max = X[:, 0].min() - marge, X[:, 0].max() + marge
    x2_min, x2_max = X[:, 1].min() - marge, X[:, 1].max() + marge
    xx, yy = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                          np.linspace(x2_min, x2_max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(titol, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def crea_models():
    """Un joc nou dels cinc models, sempre amb els mateixos hiperparàmetres."""
    return {
        "k-NN": KNeighborsClassifier(n_neighbors=5),
        "Arbre": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Bosc": RandomForestClassifier(random_state=RANDOM_STATE),
        "Regressió logística": LogisticRegression(),
        "SVM (rbf)": SVC(kernel="rbf", random_state=RANDOM_STATE),
    }

## Exercici 1 — Fabricar i mirar

Genera tres conjunts de dades, tots amb dues columnes i `random_state=42`:

- `Xb, yb`: `make_blobs` amb `n_samples=300` i `centers=3`.
- `Xm, ym`: `make_moons` amb `n_samples=300` i `noise=0.2`.
- `Xc, yc`: `make_circles` amb `n_samples=300`, `noise=0.08` i `factor=0.4`.

Dibuixa'ls en una figura d'1 fila x 3 columnes (un `scatter` per conjunt,
pintat pel seu `y`).

Abans de continuar, **aposta**: per a cada conjunt i cada model dels cinc,
omple `apostes` amb `"bé"` o `"malament"` segons si creus que aquell model
hi farà bona feina. Fes-ho només mirant els dibuixos, sense entrenar res
encara — a l'exercici 7 es comprova.

**Com saps que ho has fet bé**: els tres dibuixos mostren formes clarament
diferents (grups separats, dues llunes entrellaçades, un cercle dins de
l'altre), i les 15 caselles de `apostes` tenen `"bé"` o `"malament"`, no
`None`.

In [ ]:
# Xb, yb = make_blobs(n_samples=300, centers=3, random_state=RANDOM_STATE)
# Xm, ym = make_moons(n_samples=300, noise=0.2, random_state=RANDOM_STATE)
# Xc, yc = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=RANDOM_STATE)

# figura 1x3: un scatter per conjunt (colors segons y)

# aposta abans de veure cap número
apostes = {
    "blobs":   {"k-NN": None, "Arbre": None, "Bosc": None, "Regressió logística": None, "SVM (rbf)": None},
    "moons":   {"k-NN": None, "Arbre": None, "Bosc": None, "Regressió logística": None, "SVM (rbf)": None},
    "circles": {"k-NN": None, "Arbre": None, "Bosc": None, "Regressió logística": None, "SVM (rbf)": None},
}

## Exercici 2 — Blobs, el cas fàcil

Separa `Xb, yb` en entrenament i test (`test_size=0.3`,
`random_state=RANDOM_STATE`, `stratify=yb`). Entrena els cinc models de
`crea_models()` amb l'entrenament.

Dibuixa les cinc fronteres amb `dibuixa_frontera`, en una figura d'1x5, amb
la precisió de test a cada títol. Guarda les precisions en un diccionari
`precisions_blobs` (nom del model -> precisió).

**Com saps que ho has fet bé**: les cinc precisions superen el 95 % — amb
tres grups tan separats gairebé no hi ha marge per equivocar-se. El que
importa aquí no és qui guanya, sinó **la forma** de cada frontera: la
recta, les escales, els contorns lisos...

In [ ]:
# Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(...)

# entrena els cinc models amb crea_models() i guarda'ls en un diccionari

# figura 1x5: dibuixa_frontera(ax, model, Xb_tr, yb_tr, titol=f"{nom}\n{precisio:.1%}")

precisions_blobs = {}  # nom del model -> precisió de test

## Exercici 3 — Moons, el cas que trenca la recta

Repeteix exactament l'exercici 2 però amb `Xm, ym` (les llunes). Guarda les
precisions a `precisions_moons`.

Després, en una cel·la Markdown, escriu dues o tres frases explicant **què
veus a la frontera de la regressió logística** que no veus a les altres
quatre, i per què passa (pensa en la forma de les dades i en la forma que
sap dibuixar cada model).

**Com saps que ho has fet bé**: la Regressió logística queda clarament per
sota dels altres quatre models (una diferència d'uns 10 punts o més), i la
seva frontera —una recta— talla les dues llunes sense seguir-ne la forma.

In [ ]:
# igual que l'exercici 2 però amb Xm, ym

precisions_moons = {}  # nom del model -> precisió de test

*(Escriu aquí la teva explicació sobre la frontera de la regressió
logística a moons.)*

## Exercici 4 — L'escala de l'arbre

Amb `Xm_tr, ym_tr` (l'entrenament de moons de l'exercici 3), entrena quatre
`DecisionTreeClassifier` amb `max_depth` 1, 3, 5 i 20
(`random_state=RANDOM_STATE` a tots). Dibuixa només la frontera de
**l'arbre** (amb `dibuixa_frontera`) en una figura d'1x4, amb la
profunditat, el nombre de fulles (`model.get_n_leaves()`) i la precisió de
test a cada títol.

**Com saps que ho has fet bé**: amb `max_depth=1` la frontera és un únic
tall (2 fulles, just per sobre del 75 % de test): l'arbre amb prou feines
separa res. A mesura que puja la profunditat, els rectangles es
multipliquen (6, 12, 15 fulles) i la frontera s'ajusta cada cop més a la
forma de les llunes. Fixa't que aquí el nombre de fulles **no explota**
(de 12 a 15, no d'12 a centenars): amb només dues columnes i poc soroll,
l'arbre no troba gaire marge per memoritzar. Amb dades més sorolloses
(exercici 5) o amb columnes irrellevants (exercici 6) sí que ho fa.

In [ ]:
# per a profunditat in [1, 3, 5, 20]:
#     entrena un DecisionTreeClassifier(max_depth=profunditat, random_state=RANDOM_STATE)
#     amb Xm_tr, ym_tr

# figura 1x4: dibuixa_frontera(ax, arbre, Xm_tr, ym_tr,
#              titol=f"profunditat={p}\n{n_fulles} fulles, {precisio:.1%}")

## Exercici 5 — Soroll

Genera tres versions de moons amb `noise` 0.05, 0.2 i 0.4 (300 mostres,
`random_state=RANDOM_STATE` a totes). Per a cada `noise`, separa en
entrenament/test, entrena els cinc models de `crea_models()` i guarda la
precisió de test de cadascun. Munta-ho en una taula (files = `noise`,
columnes = model) amb un `pd.DataFrame`.

**Com saps que ho has fet bé**: totes les columnes de la taula baixen (o
com a mínim no pugen) a mesura que puja el `noise` — per exemple, el k-NN
passa d'un 100 % amb `noise=0.05` a un 82,2 % amb `noise=0.4`. Cap columna
es queda immune: **cap model arregla unes dades dolentes**.

In [ ]:
resultats_soroll = {}  # noise -> {nom_model: precisio}

# per a noise in [0.05, 0.2, 0.4]:
#     genera make_moons(n_samples=300, noise=noise, random_state=RANDOM_STATE)
#     separa en train/test, entrena els cinc models, guarda les precisions

# taula_soroll = pd.DataFrame(resultats_soroll).T

## Exercici 6 — Fabricar un problema a mida

Genera `Xcl, ycl` amb `make_classification(n_samples=300, n_features=20,
n_informative=5, n_redundant=0, n_repeated=0, n_classes=2,
random_state=RANDOM_STATE)`: 20 columnes, de les quals només 5 porten
informació de veritat; les altres 15 són soroll. Amb 20 columnes ja no es
pot dibuixar cap frontera.

Separa en entrenament/test i compara un `DecisionTreeClassifier` sol
contra un `RandomForestClassifier`, tots dos amb `random_state=RANDOM_STATE`
i sense limitar la profunditat. Imprimeix les dues precisions de test.

**Com saps que ho has fet bé**: el bosc guanya clarament a l'arbre sol (uns
8 punts de diferència: 85,6 % contra 77,8 % en aquestes dades). Amb Iris
(4 columnes, totes informatives) l'arbre sol ja anava molt bé i el bosc
gairebé no hi guanyava marge; aquí, amb 15 columnes de soroll de mig, és
**al revés**: l'arbre sol es deixa enredar per alguna columna irrellevant i
el bosc, votant entre molts arbres diferents, ho compensa.

In [ ]:
# Xcl, ycl = make_classification(n_samples=300, n_features=20, n_informative=5,
#                                 n_redundant=0, n_repeated=0, n_classes=2,
#                                 random_state=RANDOM_STATE)

# separa en train/test

# arbre = DecisionTreeClassifier(random_state=RANDOM_STATE) -> fit -> score
# bosc  = RandomForestClassifier(random_state=RANDOM_STATE) -> fit -> score

## Exercici 7 — Tancament

Recupera les apostes de l'exercici 1. Per completar la comparació et falten
les precisions reals sobre `circles`: entrena els cinc models de
`crea_models()` sobre `Xc, yc` (mateix `train_test_split` que als exercicis
2 i 3) i guarda-les a `precisions_circles`.

Considera que una aposta era **encertada** si vas dir `"bé"` i la precisió
real és `>= 0.85`, o si vas dir `"malament"` i la precisió real és `< 0.85`.
Recorre `apostes`, `precisions_blobs`, `precisions_moons` i
`precisions_circles`, i imprimeix, per a cada conjunt i cada model, si
l'aposta va ser un encert o un error.

**Com saps que ho has fet bé**: totes les apostes sobre `blobs` haurien de
sortir encertades (els cinc models hi van bé). A `moons` i `circles`,
l'única aposta que hauria de fallar amb el llindar del 85 % és la que vas
fer sobre la regressió logística, si vas apostar que hi aniria bé.

In [ ]:
# Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, ...)
# entrena els cinc models i omple precisions_circles

precisions_circles = {}

LLINDAR = 0.85

# per a cada conjunt (blobs, moons, circles) i cada model:
#     compara apostes[conjunt][model] amb la precisió real (>= LLINDAR -> "bé")
#     imprimeix si l'aposta va ser un encert o un error

## Resum

- Amb dues columnes es pot **dibuixar** la frontera de decisió: la forma
  delata el model (recta, escales, corbes, fronteres irregulars).
- La **regressió logística** només dibuixa rectes; falla clarament davant
  de dades que no se separen linealment (moons, circles).
- L'**arbre** dibuixa escales que es multipliquen amb la profunditat.
- El **bosc** suavitza les escales de l'arbre, i guanya més clarament com
  més soroll o columnes irrellevants hi ha (a l'inrevés del que passava a
  Iris).
- El **soroll** degrada tots els models per igual: cap algorisme el pot
  compensar.